# Esercizio: addestra il tuo mini Transformer decoder-only (Keras)

## Cosa stiamo studiando
Lavoriamo su un **Transformer decoder-only** (la stessa famiglia di architetture usata da GPT): un modello che legge una sequenza di parole e impara a prevedere, per ogni posizione, **la parola successiva** — guardando solo il testo che la precede, mai quello che segue. Lo abbiamo già visto nel notebook `05_Transformer_e_LLM_con_addestramento.ipynb`: è la riga `use_causal_mask=True` dentro la Multi-Head Attention che rende il modello decoder-only e non encoder.

Qui non costruisci l'architettura del modello: te la forniamo già pronta e funzionante (la classe `MiniGPT`). Il tuo lavoro è imparare a **preparare i dati e usare Keras per addestrarla davvero**, dall'inizio alla fine.

## Obiettivo dell'esercizio
Alla fine dovrai avere un modello che:
1. è stato **addestrato per davvero** (durante il training la loss deve scendere, segno che il modello sta imparando);
2. sa **generare nuove parole** plausibili a partire da una frase seed, usando ciò che ha imparato dal tuo corpus.

## Passi richiesti (in breve — la guida completa e dettagliata è più sotto)
1. Scrivere un piccolo corpus di frasi e trasformarlo in numeri (tokenizzazione + vocabolario).
2. Costruire le sequenze di input/target per il compito di "next-token prediction".
3. Creare un'istanza del modello `MiniGPT` già pronto.
4. Configurare il training (`compile`) e addestrare il modello (`fit`).
5. Verificare che il training abbia funzionato (grafico della loss).
6. Scrivere una funzione che generi testo col modello addestrato.

## Come è organizzato questo notebook
Sotto trovi prima il setup e l'architettura del modello, già pronti e funzionanti: non devi scriverli, li usi soltanto.

Il tuo compito è **uno solo**: scrivere tu, da zero, tutto il codice per addestrare il modello, seguendo la **guida passo-passo** più in basso. La guida ti dice cosa fare in italiano, passo dopo passo — non contiene codice: tocca a te tradurre ogni passo in Python.

Le celle di codice vuote dopo la guida sono divise solo per comodità (per poter eseguire ed eventualmente correggere un pezzo alla volta), ma fanno tutte parte dello stesso esercizio, in ordine.


## Setup e architettura (già pronti, non serve scriverli)

Le prossime due celle importano le librerie e definiscono la classe `MiniGPT` che userai. Eseguile e basta.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.random.set_seed(42)
np.random.seed(42)
random.seed(42)

print("Setup completato.")
print(f"TensorFlow version: {tf.__version__}")


In [ ]:
class PositionalEncoding(layers.Layer):
    """Aggiunge informazione sulla posizione di ciascun token."""

    def __init__(self, max_len=5000, d_model=512):
        super().__init__()
        self.d_model = d_model
        pe = np.zeros((max_len, d_model))
        position = np.arange(0, max_len)[:, np.newaxis]
        div_term = np.exp(np.arange(0, d_model, 2) * -(np.log(10000.0) / d_model))
        pe[:, 0::2] = np.sin(position * div_term)
        pe[:, 1::2] = np.cos(position * div_term)
        self.pe = tf.constant(pe, dtype=tf.float32)

    def call(self, x):
        seq_len = tf.shape(x)[1]
        return x + self.pe[:seq_len, :]


class DecoderBlock(layers.Layer):
    """Un blocco decoder: Multi-Head Attention causale + Feed-Forward.

    use_causal_mask=True e' cio' che rende questo blocco decoder-only
    (stile GPT): ogni token vede solo i token che lo precedono.
    """

    def __init__(self, d_model, num_heads, dff, dropout_rate=0.1):
        super().__init__()
        self.mha = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=d_model // num_heads,
            dropout=dropout_rate,
        )
        self.ffn = tf.keras.Sequential([
            layers.Dense(dff, activation='relu'),
            layers.Dense(d_model),
        ])
        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = layers.Dropout(dropout_rate)
        self.dropout2 = layers.Dropout(dropout_rate)

    def call(self, x, training=False):
        attn_output = self.mha(
            x, x, x,
            training=training,
            use_causal_mask=True,
        )
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(x + attn_output)

        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        out2 = self.layernorm2(out1 + ffn_output)
        return out2


class MiniGPT(tf.keras.Model):
    """Transformer decoder-only in miniatura, in stile GPT."""

    def __init__(self, vocab_size, d_model, num_heads, num_layers, dff,
                 max_len=1000, dropout_rate=0.1):
        super().__init__()
        self.d_model = d_model
        self.vocab_size = vocab_size
        self.embedding = layers.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(max_len, d_model)
        self.decoder_blocks = [
            DecoderBlock(d_model, num_heads, dff, dropout_rate)
            for _ in range(num_layers)
        ]
        self.final_layer = layers.Dense(vocab_size)
        self.dropout = layers.Dropout(dropout_rate)

    def call(self, x, training=False):
        x = self.embedding(x)
        x *= tf.math.sqrt(tf.cast(self.d_model, tf.float32))
        x = self.pos_encoding(x)
        x = self.dropout(x, training=training)
        for decoder_block in self.decoder_blocks:
            x = decoder_block(x, training=training)
        return self.final_layer(x)


print("Classi definite: PositionalEncoding, DecoderBlock, MiniGPT")
print("Non devi modificarle: le userai nei passi della guida qui sotto.")


## La tua guida passo-passo

Segui questi passi **in ordine**. Non c'è codice qui: leggi cosa fare e scrivilo tu in Python nelle celle vuote più sotto (sono divise a gruppi di passi solo per comodità di esecuzione).

1. Crea una variabile chiamata `testo_train`: una stringa con **almeno 8 frasi brevi**, scritte da te. Usa poche parole in totale (8-10), che si ripetono in frasi diverse (es. "il gatto dorme", "il cane corre"...): più si ripetono, più in fretta il modello riesce a imparare con un training breve.

2. Trasforma `testo_train` in una lista di singole parole, tutte minuscole (tokenizzazione). Chiama il risultato `tokens_train`.

3. Costruisci il vocabolario: l'insieme delle parole **distinte** presenti in `tokens_train` (senza ripetizioni), ordinato. Chiamalo `vocab_train`. Crea poi due dizionari: `word_to_id` (da parola a numero) e `id_to_word` (da numero a parola). Salva anche il numero di parole distinte in `vocab_size_train`.

4. Usa `word_to_id` per trasformare `tokens_train` in una lista di numeri. Chiamala `token_ids_train`.

5. Scegli una lunghezza di sequenza — una variabile `seq_len_train` (per esempio un numero tra 3 e 6) — e costruisci due liste, `X_train` e `y_train`: scorrendo `token_ids_train`, ogni elemento di `X_train` è una finestra di `seq_len_train` id consecutivi, e il corrispondente elemento di `y_train` è la stessa finestra ma spostata di una posizione in avanti (il token che segue ciascuno di quelli in input: è il compito di "next-token prediction"). Alla fine trasforma entrambe le liste in array numpy.

6. Crea un modello `MiniGPT` (la classe è già definita sopra: non riscriverla, usala) e chiamalo `mini_gpt_train`. Usa `vocab_size_train` come vocabolario. Scegli dimensioni piccole per un training veloce: `d_model` tra 32 e 128, `num_heads` che **divida esattamente** `d_model`, `num_layers` pari a 1 o 2, `dff` circa il doppio di `d_model`, e `max_len` uguale a `seq_len_train`.

7. "Compila" `mini_gpt_train` (metodo `.compile()`): scegli un optimizer adatto a reti neurali (in classe abbiamo usato Adam), una loss adatta a un compito di classificazione con target espressi come **numeri interi** e non come vettori one-hot (in classe: `SparseCategoricalCrossentropy`, con `from_logits=True` perché il modello non applica già una softmax finale), e come metrica quella che misura la frazione di predizioni corrette.

8. Allena il modello (metodo `.fit()`), passando `X_train` e `y_train`. Scegli un numero di epoche (per esempio tra 50 e 150) e una dimensione di batch (per esempio 8). Salva il risultato di `.fit()` in una variabile chiamata `history` (serve al passo successivo).

9. Usa `history.history["loss"]` per disegnare un grafico della loss lungo le epoche (con `matplotlib.pyplot`, già importato come `plt`). Se il training ha funzionato, la loss deve scendere.

10. Scrivi una funzione che generi nuovo testo a partire da una frase seed (1-2 parole del tuo vocabolario): ad ogni passo, dai in input al modello la finestra di token più recente (lunga `seq_len_train`, riusando l'ultima parola se la frase è più corta), guarda le previsioni del modello sull'**ultima posizione**, prendi l'indice della parola con il punteggio più alto (questo si chiama "greedy decoding": in TensorFlow, la funzione che restituisce l'indice del valore massimo di un tensore è `tf.argmax`) e aggiungila alla sequenza generata. Ripeti per un numero di passi a tua scelta, poi prova la funzione con un seed che sceglierai tra le parole del tuo vocabolario.

**Come capire se hai fatto bene:** se tutti i passi sono corretti, alla fine il grafico della loss scenderà e la funzione di generazione produrrà parole che "assomigliano" al tuo corpus (non frasi a caso).


In [ ]:
# Scrivi qui il codice per i punti 1-5 della guida:
# corpus (testo_train) -> tokenizzazione (tokens_train) -> vocabolario
# (vocab_train, word_to_id, id_to_word, vocab_size_train) -> id numerici
# (token_ids_train) -> sequenze di training (X_train, y_train).



In [ ]:
# Scrivi qui il codice per il punto 6 della guida:
# crea il modello mini_gpt_train (istanza di MiniGPT).



In [ ]:
# Scrivi qui il codice per i punti 7-8 della guida:
# compila mini_gpt_train (.compile) e allenalo (.fit), salvando il
# risultato in una variabile chiamata history.



In [ ]:
# Scrivi qui il codice per il punto 9 della guida:
# grafico della loss lungo le epoche, usando history.history["loss"].



In [ ]:
# Scrivi qui il codice per il punto 10 della guida:
# funzione di generazione autoregressiva (greedy, con tf.argmax) e
# una prova con un seed a tua scelta tra le parole del tuo vocabolario.



## Domanda finale (rispondi con 2-3 frasi in questa cella)

Guarda il testo che hai generato al punto 10: ha senso rispetto al tuo corpus? Cosa cambierebbe secondo te se avessi usato meno epoche, o un corpus con frasi meno ripetitive?

*Risposta:*
